# Pruning a pretrained object detector

SSDlite320 MobileNetV3-Large is an object detector made for phones: 3.4 million parameters, trained by torchvision on COCO (80 object classes). We'll shrink it by pruning, and measure what we gain and what we lose:

1. Measuring quality: the mAP
2. Measuring cost: parameters, size, latency
3. Unstructured pruning
4. Structured pruning with Torch-Pruning
5. Retraining the pruned model
6. The loss, class by class

This notebook is meant for a GPU: *Runtime > Change runtime type > T4 GPU*. Without a GPU, it still runs, on a smaller scale: 60 images instead of 943, and a single retraining epoch, from which nothing can be concluded.

## Setup

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh coco-val2017-open
%pip install -q "torchmetrics[detection]" torch-pruning

In [ ]:
import copy
import gzip
import io
import json
import statistics
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch_pruning as tp
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, decode_image
from torchvision.models.detection import (
  SSDLite320_MobileNet_V3_Large_Weights,
  ssdlite320_mobilenet_v3_large,
)
from torchvision.transforms import v2
from torchvision.utils import draw_bounding_boxes

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Without a GPU: 60 images instead of 943, and a single retraining epoch
N_IMAGES = 943 if GPU else 60
EPOCHS = 10 if GPU else 1
LEARNING_RATE = 3e-4
torch.manual_seed(0)
# A warning of Torch-Pruning's with no consequence on this model
warnings.filterwarnings("ignore", "Maximum recursive depth", module="torch_pruning")
print(
  "GPU" if GPU else "No GPU: 60 images and one epoch, enough to check the code only"
)

The data (the `coco-val2017-open` folder of shuuchuu/datasets, 156 MB): 943 images of COCO val2017 (Lin et al., 2014), the first 1,000 by id whose photo is under the CC BY 2.0 or CC BY-SA 2.0 license and that hold at least one object, less the 57 whose author has restricted the license since, with their annotations (under the CC BY 4.0 license, like all of COCO's). Each photo's author, license and Flickr page are in `coco-val2017-open/credits.csv`. The model has never seen these images, torchvision trained it on COCO train2017: the first half is used to retrain it, the second to test it.

In [ ]:
DATA = Path("coco-val2017-open")
coco = json.loads((DATA / "annotations.json").read_text())
annotations = defaultdict(list)
for annotation in coco["annotations"]:
  annotations[annotation["image_id"]].append(annotation)
infos = sorted(coco["images"], key=lambda info: info["id"])[:N_IMAGES]
images_dir = DATA / "images"
print(
  len(infos),
  "images,",
  sum(len(annotations[i["id"]]) for i in infos),
  "annotated objects",
)

The model resizes every image to 320 × 320 before processing it: we do it here once and for all, boxes included, which keeps the images in memory (300 MB) and speeds up training. Stretching an image doesn't change the IoU of two boxes, so it doesn't change the mAP.

Each item is an image and its target: the `boxes` as (x1, y1, x2, y2), the `labels` (COCO's ids, from 1 to 90) and `iscrowd` (1 for a box around a crowd, which COCO's mAP counts neither as found nor as missed). Training doesn't use these crowd boxes, and flips one image in two horizontally.

In [ ]:
SIZE = 320


def load(info: dict, crowds: bool) -> tuple[tv_tensors.Image, dict]:
  image = decode_image(str(images_dir / info["file_name"]), mode=ImageReadMode.RGB)
  sx, sy = SIZE / info["width"], SIZE / info["height"]
  kept = [
    a
    for a in annotations[info["id"]]
    if min(a["bbox"][2:]) >= 1 and (crowds or not a["iscrowd"])
  ]
  # COCO's boxes are (x, y, width, height), in pixels
  boxes = [
    [x * sx, y * sy, (x + w) * sx, (y + h) * sy]
    for x, y, w, h in (a["bbox"] for a in kept)
  ]
  target = {
    "boxes": tv_tensors.BoundingBoxes(
      torch.tensor(boxes).reshape(-1, 4), format="XYXY", canvas_size=(SIZE, SIZE)
    ),
    "labels": torch.tensor([a["category_id"] for a in kept], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in kept], dtype=torch.int64),
  }
  return tv_tensors.Image(v2.functional.resize(image, [SIZE, SIZE])), target


class Detection(Dataset):
  def __init__(self, items: list, train: bool = False) -> None:
    self.items = items
    flip = [v2.RandomHorizontalFlip()] if train else []
    self.transform = v2.Compose([*flip, v2.ToDtype(torch.float32, scale=True)])

  def __len__(self) -> int:
    return len(self.items)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, dict]:
    return self.transform(*self.items[i])


def collate(batch: list) -> tuple:
  return tuple(zip(*batch))


half = len(infos) // 2
train_set = Detection([load(info, crowds=False) for info in infos[:half]], train=True)
test_set = Detection([load(info, crowds=True) for info in infos[half:]])
train_loader = DataLoader(train_set, batch_size=16, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_set, batch_size=16, collate_fn=collate)
print(len(train_set), "training images,", len(test_set), "test images")

The model takes a list of images (3 × H × W tensors of floats between 0 and 1) and, in evaluation mode, returns a dictionary for each: `boxes`, `scores` and `labels`, up to 300 detections sorted by score. Here are those with a score above 0.5 on a test image:

In [ ]:
WEIGHTS = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
categories = WEIGHTS.meta["categories"]  # the name of each class id
model = ssdlite320_mobilenet_v3_large(weights=WEIGHTS).to(DEVICE).eval()

image, target = test_set[0]
with torch.inference_mode():
  (prediction,) = model([image.to(DEVICE)])
keep = prediction["scores"] > 0.5
drawn = draw_bounding_boxes(
  v2.functional.to_dtype(image, torch.uint8, scale=True),
  prediction["boxes"][keep].cpu(),
  labels=[categories[i] for i in prediction["labels"][keep].tolist()],
  colors="red",
  width=2,
)
plt.imshow(drawn.permute(1, 2, 0))
plt.axis("off")
plt.show()

## 1. Measuring quality: the mAP

COCO's mAP (*mean average precision*): for each class, the area under the precision-recall curve of its detections, a detection being right if its IoU with a box of the same class is above a threshold; then the mean over the classes, and over the IoU thresholds from 0.5 to 0.95. `map_50` is the one of the 0.5 threshold alone.

Write `evaluate(model, loader, class_metrics=False)`, which puts the model in evaluation mode, predicts the images of `loader` under `torch.inference_mode()`, and returns the result of a `MeanAveragePrecision` updated batch by batch. Measure the pretrained model's mAP on `test_loader`. torchvision reports 21.3% (0.213) on the whole of COCO val2017: do you get the same on these 500 images?

**Useful documentation**

- [`MeanAveragePrecision`](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html)
- [`ssdlite320_mobilenet_v3_large`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html)
- [`torch.inference_mode`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

In [ ]:
# Your code here
def evaluate(
  model: nn.Module, loader: DataLoader, class_metrics: bool = False
) -> dict: ...

### Additional hints

The calls and parameters to use:

- `model.eval()`, then everything else under `with torch.inference_mode():`
- `MeanAveragePrecision(iou_type="bbox", class_metrics=class_metrics)`: boxes as (x1, y1, x2, y2), its default format; `class_metrics=True` adds each class's average precision (`map_per_class`, for exercise 6)
- `metric.warn_on_many_detections = False`: the model returns up to 300 detections per image, COCO's mAP only counts the best 100, and `MeanAveragePrecision` warns about it at each batch
- each batch of `loader` is a tuple of images and a tuple of targets: `for images, targets in loader:`
- `model([image.to(DEVICE) for image in images])` returns the list of predictions
- `metric.update(predictions, targets)`: two lists of dictionaries, on the same device; bring the predictions back to the CPU, where the targets are (`{k: v.cpu() for k, v in p.items()}`)
- `metric.compute()` returns a dictionary of tensors: `float(results["map"])`, `float(results["map_50"])`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def evaluate(model: nn.Module, loader: DataLoader, class_metrics: bool = False) -> dict:
  model.eval()
  metric = MeanAveragePrecision(iou_type="bbox", class_metrics=...)
  metric.warn_on_many_detections = False
  with torch.inference_mode():
    for images, targets in loader:
      predictions = model(...)
      metric.update([{k: v.cpu() for k, v in p.items()} for p in predictions], ...)
  return metric.compute()


base_metrics = evaluate(model, test_loader)
print(
  f"mAP {float(base_metrics['map']):.3f}, mAP@0.5 {float(base_metrics['map_50']):.3f}"
)
```

### Solution

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, class_metrics: bool = False) -> dict:
  model.eval()
  metric = MeanAveragePrecision(iou_type="bbox", class_metrics=class_metrics)
  # COCO's mAP only counts the best 100 detections per image
  metric.warn_on_many_detections = False
  with torch.inference_mode():
    for images, targets in loader:
      predictions = model([image.to(DEVICE) for image in images])
      metric.update([{k: v.cpu() for k, v in p.items()} for p in predictions], targets)
  return metric.compute()


base_metrics = evaluate(model, test_loader)
print(
  f"mAP {float(base_metrics['map']):.3f}, mAP@0.5 {float(base_metrics['map_50']):.3f}"
)

## 2. Measuring cost: parameters, size, latency

Write `profile(model)`, which returns a dictionary:

- `params_M`: the model's number of parameters, in millions;
- `size_MB`: the size of the file of its weights (`torch.save(model.state_dict(), ...)`), in MB;
- `gzip_MB`: the size of that file compressed with gzip, in MB;
- `latency_ms`: the median latency of a prediction of a single image (batch of 1), in milliseconds, **on the CPU**: SSDlite is made for phones and embedded processors, its target hardware. Predict `image` 25 times with a copy of the model on the CPU, and ignore the first 5 measurements (the warm-up).

A latency is measured on an idle machine: on Colab, it varies by a few percent from one run to the next.

**Useful documentation**

- [`torch.save`](https://docs.pytorch.org/docs/stable/generated/torch.save.html)
- [`io.BytesIO`](https://docs.python.org/3/library/io.html#io.BytesIO)
- [`gzip.compress`](https://docs.python.org/3/library/gzip.html#gzip.compress)
- [`copy.deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)
- [`statistics.median`](https://docs.python.org/3/library/statistics.html#statistics.median)

In [ ]:
image = test_set[0][0]  # the image of the latency measurements


# Your code here
def profile(model: nn.Module) -> dict: ...

### Additional hints

The calls and parameters to use:

- `sum(p.numel() for p in model.parameters())`: the number of parameters, 3.44 million for the pretrained model
- `buffer = io.BytesIO()`, then `torch.save(model.state_dict(), buffer)`: the file in memory; `buffer.getvalue()` gives its bytes
- `len(gzip.compress(data))`: the compressed size
- `cpu_model = copy.deepcopy(model).cpu().eval()`: a copy, so as not to move the model itself
- `start = time.perf_counter()`, then `cpu_model([image])`, then `time.perf_counter() - start`, in seconds, under `torch.inference_mode()`
- `statistics.median(times[5:])`: the median, without the warm-up

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def profile(model: nn.Module) -> dict:
  buffer = io.BytesIO()
  torch.save(model.state_dict(), buffer)
  cpu_model = copy.deepcopy(model).cpu().eval()
  times = []
  with torch.inference_mode():
    for _ in range(25):
      start = time.perf_counter()
      ...
      times.append(time.perf_counter() - start)
  return {
    "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    "size_MB": len(buffer.getvalue()) / 1e6,
    "gzip_MB": ...,
    "latency_ms": 1000 * statistics.median(...),
  }


profile(model)
```

### Solution

In [ ]:
def profile(model: nn.Module) -> dict:
  buffer = io.BytesIO()
  torch.save(model.state_dict(), buffer)
  cpu_model = copy.deepcopy(model).cpu().eval()
  times = []
  with torch.inference_mode():
    for _ in range(25):
      start = time.perf_counter()
      cpu_model([image])
      times.append(time.perf_counter() - start)
  return {
    "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    "size_MB": len(buffer.getvalue()) / 1e6,
    "gzip_MB": len(gzip.compress(buffer.getvalue())) / 1e6,
    "latency_ms": 1000 * statistics.median(times[5:]),
  }


profile(model)

The table of results, which each exercise adds a row to:

In [ ]:
results = {}


def record(name: str, model: nn.Module, metrics: dict) -> pd.DataFrame:
  """Add the model's row to the table, and show the table."""
  results[name] = {
    "mAP": float(metrics["map"]),
    "mAP@0.5": float(metrics["map_50"]),
    **profile(model),
  }
  return pd.DataFrame(results).T.round(3)


record("pretrained", model, base_metrics)

## 3. Unstructured pruning

On a copy of the model (`unstructured`), set to zero half of the weights of the `nn.Conv2d` and `nn.Linear` layers, those of smallest magnitude over the whole network (`prune.global_unstructured`, as in the slides), then make the pruning permanent (`prune.remove`). Print the share of zero weights, then the model's row in the table. What has changed: the mAP, the number of parameters, the file size, its compressed size, the latency?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `unstructured = copy.deepcopy(model)`: the pretrained model stays intact
- `[(m, "weight") for m in unstructured.modules() if isinstance(m, nn.Conv2d | nn.Linear)]`: the weights to prune
- `prune.global_unstructured(weights, pruning_method=prune.L1Unstructured, amount=0.5)`: the threshold is common to the whole network, so some layers lose more than half of their weights, others less
- `prune.remove(module, "weight")` for each: the mask becomes permanent
- the share of zeros: the number of zero weights (`(m.weight == 0).sum().item()`) over the number of weights (`m.weight.numel()`), 50% expected
- `record("unstructured 50%", unstructured, evaluate(unstructured, test_loader))`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
unstructured = copy.deepcopy(model)
kinds = nn.Conv2d | nn.Linear
weights = [(m, "weight") for m in unstructured.modules() if isinstance(m, kinds)]
prune.global_unstructured(weights, pruning_method=..., amount=...)
for module, name in weights:
  ...
zeros = sum((m.weight == 0).sum().item() for m, _ in weights)
total = sum(m.weight.numel() for m, _ in weights)
print(f"{zeros / total:.0%}")
record(..., unstructured, evaluate(unstructured, test_loader))
```

### Solution

In [ ]:
unstructured = copy.deepcopy(model)
kinds = nn.Conv2d | nn.Linear
weights = [(m, "weight") for m in unstructured.modules() if isinstance(m, kinds)]
prune.global_unstructured(weights, pruning_method=prune.L1Unstructured, amount=0.5)
for module, name in weights:
  prune.remove(module, name)
zeros = sum((m.weight == 0).sum().item() for m, _ in weights)
total = sum(m.weight.numel() for m, _ in weights)
print(f"{zeros / total:.0%} zero weights")
record("unstructured 50%", unstructured, evaluate(unstructured, test_loader))

## 4. Structured pruning with Torch-Pruning

Structured pruning removes whole channels: the tensors shrink. On another copy of the model (`structured`), remove 20% of the channels of each layer with a `tp.pruner.MetaPruner` and a `tp.importance.MagnitudeImportance(p=2)`, as in the slides (which remove half of them), protecting the detection heads (`structured.head`): their outputs, for each anchor the scores of the 91 class ids and the 4 coordinates of a box, are fixed by the task. Torch-Pruning follows the model's computations on an example input: here, a list of one 3 × 320 × 320 image on `DEVICE`. Show the model's row in the table: what are the mAP, the size and the latency?

**Useful documentation**

- [Torch-Pruning](https://github.com/VainF/Torch-Pruning)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `structured = copy.deepcopy(model)`
- `example = [torch.rand(3, 320, 320, device=DEVICE)]`: the model takes a list of images
- `tp.pruner.MetaPruner(structured, example, importance=tp.importance.MagnitudeImportance(p=2), pruning_ratio=0.2, ignored_layers=[structured.head])`: the channels of smallest L2 norm go, 20% of those of each layer
- `pruner.step()` removes them, once and for all; in the heads, Torch-Pruning also removes the input channels that matched the removed channels: only their outputs are protected
- `record("structured 20%", structured, evaluate(structured, test_loader))`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
structured = copy.deepcopy(model)
example = [torch.rand(3, 320, 320, device=DEVICE)]
pruner = tp.pruner.MetaPruner(
  structured,
  example,
  importance=...,
  pruning_ratio=...,
  ignored_layers=[...],
)
pruner.step()
record(..., structured, evaluate(structured, test_loader))
```

### Solution

In [ ]:
structured = copy.deepcopy(model)
example = [torch.rand(3, 320, 320, device=DEVICE)]
pruner = tp.pruner.MetaPruner(
  structured,
  example,
  importance=tp.importance.MagnitudeImportance(p=2),
  pruning_ratio=0.2,
  ignored_layers=[structured.head],
)
pruner.step()
record("structured 20%", structured, evaluate(structured, test_loader))

## 5. Retraining the pruned model

Write `train_epoch(model, loader, optimizer)`: one training epoch, which returns the mean loss. In training mode (`model.train()`), the detector takes the images **and** their targets, and returns its losses, a dictionary: `bbox_regression` (the boxes) and `classification` (the classes); the loss to minimize is their sum. Retrain `structured` for `EPOCHS` epochs on `train_loader`, with `torch.optim.AdamW` and a learning rate of `LEARNING_RATE`, printing each epoch's loss, then show the model's row in the table. What share of the lost mAP do we win back?

**Useful documentation**

- [`torch.optim.AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

In [ ]:
# Your code here
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float: ...

### Additional hints

The calls and parameters to use:

- `model.train()` at the start of the epoch (`evaluate` puts the model back in evaluation mode)
- the images and the targets on `DEVICE`: `[image.to(DEVICE) for image in images]`, `[{k: v.to(DEVICE) for k, v in t.items()} for t in targets]`
- `losses = model(images, targets)`, then `loss = sum(losses.values())`
- `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`; `loss.item()` for the mean
- `torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)`, created once, before the loop over the epochs
- `record("structured 20%, retrained", structured, evaluate(structured, test_loader))`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float:
  model.train()
  total = 0.0
  for images, targets in loader:
    images = [image.to(DEVICE) for image in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    losses = model(...)
    loss = ...
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total += loss.item()
  return total / len(loader)


optimizer = torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
  print(epoch + 1, train_epoch(structured, train_loader, optimizer))
record(..., structured, evaluate(structured, test_loader))
```

### Solution

In [ ]:
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float:
  model.train()
  total = 0.0
  for images, targets in loader:
    images = [image.to(DEVICE) for image in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    losses = model(images, targets)
    loss = sum(losses.values())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total += loss.item()
  return total / len(loader)


optimizer = torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
  loss = train_epoch(structured, train_loader, optimizer)
  print(f"epoch {epoch + 1}: loss {loss:.3f}")
record("structured 20%, retrained", structured, evaluate(structured, test_loader))

## 6. The loss, class by class

An overall mAP can hide losses concentrated on a few classes: a pruned network forgets the rare and difficult examples first (Hooker et al., 2019, *What Do Compressed Deep Neural Networks Forget?*). Evaluate the pretrained model and the retrained model with `class_metrics=True`, then build a `DataFrame` indexed by the class name, with the number of objects of the class in the test images, the average precision (AP) before and after pruning, and their difference. Keep the classes with at least `MIN_OBJECTS` objects, and sort them by difference. Do all the classes lose as much? Are those that lose the most the rarest?

**Useful documentation**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
# Fewer objects: the class's AP depends too much on the few images drawn
MIN_OBJECTS = 20 if GPU else 1

# Your code here

### Additional hints

The calls and parameters to use:

- `evaluate(model, test_loader, class_metrics=True)` and the same for `structured`: `metrics["map_per_class"]` gives each class's AP, in the order of the ids of `metrics["classes"]`; `pd.Series(aps, index=ids)` pairs them
- the objects of each class: a `Counter` of the `labels` of the targets of `test_set.items`, without the crowd boxes (`iscrowd` at 1)
- `pd.DataFrame({"objects": ..., "AP before": ..., "AP after": ...})` aligns the three series on their ids; `classes.index = [categories[i] for i in classes.index]` replaces them with the names
- `classes[classes["objects"] >= MIN_OBJECTS].sort_values("difference")`: the class that loses the most first

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def per_class(metrics: dict) -> pd.Series:
  return pd.Series(metrics["map_per_class"].tolist(), index=metrics["classes"].tolist())


before = per_class(evaluate(model, test_loader, class_metrics=True))
after = per_class(...)
objects = Counter(
  label
  for _, target in test_set.items
  for label, crowd in zip(target["labels"].tolist(), target["iscrowd"].tolist())
  if not crowd
)
classes = pd.DataFrame(
  {"objects": pd.Series(objects), "AP before": before, "AP after": after}
)
classes["difference"] = ...
classes.index = [categories[i] for i in classes.index]
classes[classes["objects"] >= MIN_OBJECTS].sort_values(...).round(3)
```

### Solution

In [ ]:
def per_class(metrics: dict) -> pd.Series:
  return pd.Series(metrics["map_per_class"].tolist(), index=metrics["classes"].tolist())


before = per_class(evaluate(model, test_loader, class_metrics=True))
after = per_class(evaluate(structured, test_loader, class_metrics=True))
objects = Counter(
  label
  for _, target in test_set.items
  for label, crowd in zip(target["labels"].tolist(), target["iscrowd"].tolist())
  if not crowd
)
classes = pd.DataFrame(
  {"objects": pd.Series(objects), "AP before": before, "AP after": after}
)
classes["difference"] = classes["AP after"] - classes["AP before"]
classes.index = [categories[i] for i in classes.index]
classes[classes["objects"] >= MIN_OBJECTS].sort_values("difference").round(3)

## Summary

The four models side by side:

In [ ]:
pd.DataFrame(results).T.round(3)

- Which of these models would you deploy on a phone? What would still have to be measured, and where (on the phone itself, with its inference engine)?
- Removing half of the channels, as in the slides, brings this detector's mAP down to almost 0, and a few epochs on 500 images win almost nothing of it back: a network as compact as MobileNetV3 already has few useless channels.
- To win back more mAP: retrain longer, on COCO train2017 (118,000 images); prune gradually, in small steps separated by retraining (a pruning schedule); remove fewer channels; distill the original model into the pruned one.
- Unstructured pruning only shrank the compressed file: what can it be useful for all the same?